# 模块概述

WtBtRunner 是 WonderTrader 回测运行器模块，负责历史数据回测环境的初始化和运行。主要包括：
- 回测环境初始化
- 历史数据回放器管理
- 多种策略模拟器支持（CTA、HFT、选股、执行器、UFT）
- 增量回测支持
- 定时任务管理

1. **主程序层**（main）：
   - main：主程序入口，提供命令行参数解析和回测程序启动
   - 负责解析命令行参数、初始化回测环境、加载配置、创建策略模拟器并启动历史数据回放
   - 是整个回测系统的可执行文件入口点

2. **回放器层**（HisDataReplayer）：
   - HisDataReplayer：历史数据回放器，负责历史数据的回放和分发
   - 管理回测时间范围、数据源、回放模式等
   - 将历史数据按时间顺序推送给注册的模拟器
   - 支持同步和异步回放模式

3. **模拟器层**（CtaMocker/HftMocker/SelMocker/ExecMocker/UftMocker）：
   - CtaMocker：CTA策略模拟器，模拟CTA策略在历史数据上的运行
   - HftMocker：HFT策略模拟器，模拟高频交易策略在历史数据上的运行
   - SelMocker：选股策略模拟器，模拟选股策略在历史数据上的运行
   - ExecMocker：执行器模拟器，模拟订单执行逻辑的回测
   - UftMocker：UFT策略模拟器，模拟极速交易策略在历史数据上的运行
   - 接收历史数据回放器推送的数据，执行策略逻辑，记录回测结果

4. **数据加载层**（IBtDataLoader）：
   - IBtDataLoader：回测数据加载器接口，定义历史数据加载的标准接口
   - 支持加载K线数据、Tick数据、复权因子等
   - 可以从本地文件系统或外部数据源加载历史数据

5. **工具支持层**（WtHelper）：
   - WtHelper：辅助工具类，提供路径管理、时间管理等工具函数
   - 支持工作目录设置、路径标准化等功能

# 层次关系图
```mermaid
graph TB
    %% 样式定义
    classDef mainClass fill:#fff9c4,stroke:#f57f17,stroke-width:2px,color:#000;
    classDef replayerClass fill:#e1f5ff,stroke:#01579b,stroke-width:3px,color:#000;
    classDef mockerClass fill:#fff3e0,stroke:#e65100,stroke-width:2px,color:#000;
    classDef loaderClass fill:#f3e5f5,stroke:#4a148c,stroke-width:2px,color:#000;
    classDef utilClass fill:#fce4ec,stroke:#880e4f,stroke-width:2px,color:#000;
    classDef interfaceClass fill:#f5f5f5,stroke:#616161,stroke-width:1px,stroke-dasharray: 5 5,color:#000;

    %% 主程序层
    subgraph MainLayer["主程序层 - 程序入口"]
        direction TB
        Main["main<br/>主程序入口<br/>• 命令行参数解析<br/>• 初始化日志系统<br/>• 加载配置文件<br/>• 创建回放器和模拟器<br/>• 启动历史数据回放"]:::mainClass
    end

    %% 回放器层
    subgraph ReplayerLayer["回放器层 - 历史数据回放"]
        direction TB
        HisDataReplayer["HisDataReplayer<br/>历史数据回放器<br/>• 历史数据加载<br/>• 数据回放控制<br/>• 时间管理<br/>• 模拟器注册<br/>• 数据分发"]:::replayerClass
    end

    %% 模拟器层
    subgraph MockerLayer["模拟器层 - 策略模拟"]
        direction TB
        CtaMocker["CtaMocker<br/>CTA策略模拟器<br/>• CTA策略回测<br/>• 增量回测支持<br/>• 成交模拟<br/>• 持仓管理"]:::mockerClass
        HftMocker["HftMocker<br/>HFT策略模拟器<br/>• HFT策略回测<br/>• Level-2数据模拟<br/>• 高频交易模拟"]:::mockerClass
        SelMocker["SelMocker<br/>选股策略模拟器<br/>• 选股策略回测<br/>• 定时任务支持<br/>• 多品种持仓模拟"]:::mockerClass
        ExecMocker["ExecMocker<br/>执行器模拟器<br/>• 订单执行模拟<br/>• 执行逻辑回测"]:::mockerClass
        UftMocker["UftMocker<br/>UFT策略模拟器<br/>• UFT策略回测<br/>• 极速交易模拟"]:::mockerClass
    end

    %% 数据加载接口层
    subgraph LoaderLayer["数据加载接口层 - 历史数据加载"]
        direction TB
        IBtDataLoader["IBtDataLoader<br/>回测数据加载器接口<br/>• 加载K线数据<br/>• 加载Tick数据<br/>• 加载复权因子<br/>• 数据转储控制"]:::interfaceClass
    end

    %% 工具支持层
    subgraph UtilLayer["工具支持层 - 辅助功能"]
        direction TB
        WtHelper["WtHelper<br/>辅助工具类<br/>• 路径管理<br/>• 时间管理<br/>• 目录创建"]:::utilClass
    end

    %% 主程序到回放器
    Main -->|"创建并初始化"| HisDataReplayer

    %% 主程序到模拟器（根据配置选择）
    Main -->|"根据配置创建"| CtaMocker
    Main -->|"根据配置创建"| HftMocker
    Main -->|"根据配置创建"| SelMocker
    Main -->|"根据配置创建"| ExecMocker
    Main -->|"根据配置创建"| UftMocker

    %% 主程序到工具层
    Main -->|"使用"| WtHelper

    %% 回放器到模拟器（注册关系）
    HisDataReplayer -->|"注册并分发数据"| CtaMocker
    HisDataReplayer -->|"注册并分发数据"| HftMocker
    HisDataReplayer -->|"注册并分发数据"| SelMocker
    HisDataReplayer -->|"注册并分发数据"| ExecMocker
    HisDataReplayer -->|"注册并分发数据"| UftMocker

    %% 回放器到数据加载器（实现关系）
    HisDataReplayer -.->|"实现"| IBtDataLoader

    %% 模拟器到数据加载器（使用关系）
    CtaMocker -.->|"使用"| IBtDataLoader
    HftMocker -.->|"使用"| IBtDataLoader
    SelMocker -.->|"使用"| IBtDataLoader
    ExecMocker -.->|"使用"| IBtDataLoader
    UftMocker -.->|"使用"| IBtDataLoader

    %% 模拟器到工具层
    CtaMocker -.->|"使用"| WtHelper
    HftMocker -.->|"使用"| WtHelper
    SelMocker -.->|"使用"| WtHelper
    ExecMocker -.->|"使用"| WtHelper
    UftMocker -.->|"使用"| WtHelper

    %% 应用样式
    class Main mainClass
    class HisDataReplayer replayerClass
    class CtaMocker,HftMocker,SelMocker,ExecMocker,UftMocker mockerClass
    class IBtDataLoader interfaceClass
    class WtHelper utilClass
```

# WtBtRunner.cpp/main
WonderTrader 回测运行器的主入口，用于历史数据回测，支持 CTA/HFT/选股/执行器/UFT 策略的回测。

1. 崩溃转储初始化（Windows）
   - 启用 MiniDump，模块名 "WtBtRunner.exe"，完整转储
   - 转储文件保存到当前工作目录

2. 命令行参数解析
   - `-c, --config`：配置文件路径（默认 `./configbt.yaml`）
   - `-l, --logcfg`：日志配置文件路径（默认 `./logcfgdt.yaml`）
   - `-h, --help`：显示帮助并退出
3. 日志系统初始化
   - 使用日志配置文件初始化日志系统
   - 若未指定，使用默认路径 `./logcfgdt.yaml`
4. 安装信号钩子
   - 捕获异常和错误（如段错误、崩溃）
   - 回调函数将错误信息记录到日志系统
5. 加载主配置文件
   - 检查配置文件是否存在
   - 使用 `WTSCfgLoader::load_from_file()` 加载配置
   - 若加载失败，返回 -1
6. 创建并初始化历史数据回放器
   - 创建 `HisDataReplayer` 实例 `replayer`
   - 调用 `replayer.init()` 初始化，传入回放器配置节点
   - 配置包括：数据存储路径、回放模式、回测时间范围、Tick 回放开关等
7. 根据配置选择并创建策略模拟器
   根据配置中的 `env.mocker` 字段选择模拟器类型：

   - CTA 策略（`mode == "cta"`）
     - 创建 `CtaMocker` 实例 
     - 初始化 CTA 策略工厂，加载策略动态库
     - 支持增量回测：若配置了 `incremental_backtest_base`，调用 `load_incremental_data()` 加载基础历史回测数据
     - 注册模拟器到历史数据回放器

   - HFT 策略（`mode == "hft"`）
     - 创建 `HftMocker` 实例
     - 初始化 HFT 策略工厂
     - 注册模拟器到历史数据回放器

   - 选股策略（`mode == "sel"`）
     - 创建 `SelMocker` 实例
     - 初始化选股策略工厂
     - 注册模拟器到历史数据回放器
     - 注册定时任务：调用 `register_task()` 注册选股触发任务（日期、时间、周期）

   - 执行器回测（`mode == "exec"`）
     - 创建 `ExecMocker` 实例
     - 调用 `init()` 初始化执行器模拟器
     - 注册模拟器到历史数据回放器

   - UFT 策略（`mode == "uft"`）
     - 创建 `UftMocker` 实例
     - 初始化 UFT 策略工厂
     - 注册模拟器到历史数据回放器
8. 准备回测环境
   - 调用 `replayer.prepare()` 准备回测环境
   - 流程包括：
     - 加载历史数据（K 线、Tick 等）
     - 初始化模拟器
     - 设置回测时间范围
     - 准备数据缓存
9. 启动历史数据回放（异步模式）
   - 调用 `replayer.run(true)` 启动回放，`true` 表示异步模式（不阻塞主线程）
   - 回放流程：
     - 按时间顺序回放历史数据
     - 将数据推送给注册的模拟器
     - 模拟器执行策略逻辑
     - 记录回测结果（成交记录、持仓记录、资金曲线等）
10. 等待用户退出
    - 打印提示信息 "press enter key to exit"
    - 调用 `getchar()` 等待用户按键（阻塞主线程）
    - 用户按回车键后继续执行
11. 停止日志系统
    - 调用 `WTSLogger::stop()` 停止日志系统
    - 程序退出